# 02 · Features derivadas y prueba de no-fuga

**Momento de la predicción.** El horario de la semana siguiente se arma el domingo por la noche, con los datos hasta ese día. Una feature es válida solo si se puede calcular en ese momento para **todas** las horas de la semana que se pronostica: calendario, feriados, personal planificado y ventas de hace **7 días o más**.

Las features se calculan en `src/pipeline.py::agregar_features` (no se copian aquí).

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")   # avisos de esquema de MLflow
import os, logging
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "1")
logging.getLogger("mlflow").setLevel(logging.ERROR)   # avisos repetidos de serialización/entorno
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

import src.pipeline as P
from src.datos import cargar_obt
from src import estilo
estilo.aplicar()

snap = cargar_obt()
df = P.agregar_features(snap.datos)
div = P.dividir(df)
print(f"train {len(div.train)} filas ({div.train.hour_date.min():%d-%m} a {div.train.hour_date.max():%d-%m}) · "
      f"prueba {len(div.test)} filas ({div.test.hour_date.min():%d-%m} a {div.test.hour_date.max():%d-%m})")

## Feature 1 · `ventana_pago` (derivada del hallazgo H4 del EDA)

`1` si el día del mes es 28–31 o 1–3, alrededor del pago de sueldos de fin de mes. Es una variable de **calendario**: existe para cualquier fecha futura. Es binaria, así que no necesita codificación (pasa tal cual).

In [ ]:
# Evidencia solo con el entrenamiento: residuo frente al promedio de su hora y día de la semana.
tr = div.train
res = tr[P.TARGET] - tr.groupby(["hour_of_day", "iso_day_of_week"])[P.TARGET].transform("mean")
en, fuera = res[tr.ventana_pago == 1], res[tr.ventana_pago == 0]
u = stats.mannwhitneyu(en, fuera, alternative="greater")
print(f"Residuo medio en ventana de pago: {en.mean():+.2f} ventas/h (n={len(en)}) · fuera: {fuera.mean():+.2f} (n={len(fuera)})")
print(f"Diferencia: {en.mean() - fuera.mean():+.2f} ventas/h ≈ {(en.mean() - fuera.mean()) / tr[P.TARGET].mean():+.1%} del promedio")
print(f"Mann-Whitney (unilateral, ventana > resto): p = {u.pvalue:.4f}")

**Lectura.** En el entrenamiento, el efecto va en la dirección esperada (+0,2 ventas/h, ≈ +3 %), pero **no es significativo** (p ≈ 0,24). Con 10 semanas solo hay 2 o 3 fines de mes, así que la potencia del test es baja. La feature se mantiene como **hipótesis de negocio con evidencia débil**, porque es barata y no tiene fuga; la ablación de abajo dice si aporta. Es una limitación del experimento: confirmarla requiere más meses de historia.

## Feature 2 · `ventas_dia_lag_7d`

Es la venta **total del mismo día de la semana anterior** (fecha − 7). Mide el nivel de ese día de la semana con 11 horas de información, en lugar de una sola. Por eso es menos ruidosa que el rezago de una hora (H5: los rezagos por hora son ruidosos y redundantes). Existe el domingo de planificación porque la fecha − 7 ya pasó. Es numérica: se imputa con la mediana, con indicador de faltante, y se estandariza dentro del Pipeline.

In [ ]:
diaria = tr.groupby("hour_date").agg(ventas=(P.TARGET, "sum"), lag=("ventas_dia_lag_7d", "first")).dropna()
rho_dia = stats.spearmanr(diaria.lag, diaria.ventas)
rho_hora = stats.spearmanr(tr.ventas_dia_lag_7d, tr[P.TARGET], nan_policy="omit")
print(f"ρ con la venta del DÍA: {rho_dia.statistic:.2f} (p={rho_dia.pvalue:.3f}) · ρ con la venta de la HORA: {rho_hora.statistic:.2f}")

**Lectura.** La venta del día de la semana pasada predice la venta del día de hoy (ρ = 0,50, p < 0,001). Su correlación con la venta de cada hora es baja (0,14) porque la hora domina la variación dentro del día. La feature aporta el **nivel del día**, y el modelo combina ese nivel con el perfil por hora.

## Codificación del resto de las variables

| Variable | Tipo | Codificación | Por qué |
|---|---|---|---|
| `hour_of_day` | categórica (11 niveles) | One-hot | La relación con la venta no es lineal (cae al almuerzo); para un modelo lineal, cada hora necesita su propio nivel |
| `iso_day_of_week` | categórica (7 niveles) | One-hot | No hay un orden natural entre sábado y domingo |
| `is_holiday`, `is_last_open_hour`, `ventana_pago` | binarias | Sin transformar (0/1) | Ya son indicadores |
| rezagos, prom. 4 semanas, `ventas_dia_lag_7d`, `staff_present` | numéricas | Mediana + indicador de faltante + `StandardScaler` | H3: faltan por estructura al inicio; la escala importa para Ridge y la MLP |

`OneHotEncoder(handle_unknown="ignore")`: si en producción aparece un nivel que no vio el entrenamiento, se codifica en ceros en vez de fallar.

In [ ]:
prep = P.construir_preprocesador().fit(tr[P.ENTRADA])
nombres = prep.get_feature_names_out()
print(f"{len(P.FEATURES)} variables de entrada → {len(nombres)} columnas tras el ColumnTransformer")
print(list(nombres))

## ¿Aportan? Ablación con la misma CV y las dos métricas

Se evalúa Ridge (el mejor candidato lineal) y RandomForest (el de árboles) con y sin cada feature.

In [ ]:
def ablacion(numericas, binarias):
    P.NUMERICAS[:], P.BINARIAS[:] = numericas, binarias
    filas = {}
    for nombre, est in [("Ridge", Ridge()), ("RandomForest", RandomForestRegressor(random_state=P.SEMILLA, n_jobs=-1))]:
        m = P.evaluar(P.construir_pipeline(est), div.train).metricas
        filas[nombre] = (m["cv_wape_mean"], m["cv_wape_std"], m["cv_costo_negocio"])
    return filas

NUM0, BIN0 = list(P.NUMERICAS), list(P.BINARIAS)
base_num = [c for c in NUM0 if c != "ventas_dia_lag_7d"]
base_bin = [c for c in BIN0 if c != "ventana_pago"]
variantes = {
    "sin derivadas": (base_num, base_bin),
    "+ ventana_pago": (base_num, BIN0),
    "+ ventas_dia_lag_7d": (NUM0, base_bin),
    "+ ambas (final)": (NUM0, BIN0),
}
filas = []
for etiqueta, (num, binr) in variantes.items():
    for modelo, (w, s, c) in ablacion(num, binr).items():
        filas.append({"variante": etiqueta, "modelo": modelo, "cv_wape": w, "cv_wape_std": s, "costo_cv_$": c})
P.NUMERICAS[:], P.BINARIAS[:] = NUM0, BIN0
abl = pd.DataFrame(filas).pivot(index="variante", columns="modelo").reindex(list(variantes))
abl.round(3)

**Lectura honesta.** En WAPE, las dos features cambian el resultado en menos de una centésima, muy por debajo de la desviación entre pliegues (~0,035): **en la métrica técnica no hay mejora demostrable**. Donde sí se nota es en el costo de negocio: con las dos features, el costo de los pliegues baja en los dos modelos. La razón es que ayudan a anticipar los días de más venta (fin de mes y semanas fuertes), que son los que producen horas sobrecargadas. Se mantienen por eso, y el memo no les atribuye una mejora de WAPE.

## Prueba de que no hay fuga

1. Ninguna columna de `FUGA` está entre los predictores.
2. **Prueba de truncamiento:** si se recalculan las features usando solo los datos hasta una fecha de corte *t*, las features de la semana siguiente (*t*+1 … *t*+7) deben salir **idénticas** a las calculadas con toda la tabla. Si alguna feature usara el futuro, cambiaría.
3. Las transformaciones que **aprenden** de los datos (mediana, escala, categorías) se ajustan dentro del `Pipeline`, en cada pliegue, solo con el entrenamiento de ese pliegue.

In [ ]:
assert not set(P.FUGA) & set(P.FEATURES), "hay columnas con fuga entre los predictores"
assert P.TARGET not in P.FEATURES

completo = P.agregar_features(snap.datos)
cortes = pd.date_range("2026-09-06", "2026-11-01", freq="7D")   # domingos de planificación
for corte in cortes:
    # La semana a pronosticar no tiene ventas todavía: se borran antes de calcular las features.
    pasado = snap.datos.copy()
    futuro = pasado.hour_date > corte
    pasado.loc[futuro, P.TARGET] = np.nan
    pasado = pasado[pasado.hour_date <= corte + pd.Timedelta(days=7)]
    trunc = P.agregar_features(pasado)
    semana = trunc.hour_date > corte
    a = trunc.loc[semana, P.FEATURES].reset_index(drop=True)
    b = completo.loc[completo.hour_date.between(corte + pd.Timedelta(days=1), corte + pd.Timedelta(days=7)),
                     P.FEATURES].reset_index(drop=True)
    pd.testing.assert_frame_equal(a, b, check_dtype=False)
print(f"OK: en {len(cortes)} domingos de planificación, las features de la semana siguiente no cambian al ocultar sus ventas.")

In [ ]:
# Los rezagos de la OBT (Spark, PSet 2) también se verifican: lag_7d = venta de la misma hora 7 días antes.
ref = df.set_index(["hour_date", "hour_of_day"])[P.TARGET]
clave = list(zip(df.hour_date - pd.Timedelta(days=7), df.hour_of_day))
esperado = pd.Series([ref.get(k, np.nan) for k in clave], index=df.index)
assert np.allclose(esperado, df.sales_lag_7d, equal_nan=True)
print("OK: sales_lag_7d coincide con la venta de la misma hora 7 días antes en todas las filas.")